In [1]:
import requests
import numpy as np
import pandas as pd
import pynapple as nap

Download data and import file into pandas

In [2]:
url = "https://uni-bonn.sciebo.de/public.php/webdav/YTgrGyB5MekRP2n"
print("Downloading Data ...")
response = requests.get(("/").join(url.split("/")[:-1]), auth=(url.split("/")[-1], "ibots"))
response.raise_for_status()
with open("flash_spikes.parquet", "wb") as file:
    file.write(response.content)
print("Done!")

Done!


In [4]:
df = pd.read_parquet("flash_spikes.parquet")
df

,unit_id,brain_area,spike_time
0,951031334,LM,1276.297339
1,951031154,LM,1276.298206
2,951031243,LM,1276.298739
3,951021543,PM,1276.299007
4,951031253,LM,1276.301272
...,...,...,...
457701,951021607,PM,1572.790671
457702,951039252,RL,1572.792065
457703,951016491,AM,1572.792470
457704,951031263,LM,1572.793746


To convert the spike times to a `TsGroup`, we have to extract the spike times for every unit into a dictionary of `Ts` objects where the keys are the unit IDs.

In [ ]:
spikes = {
    unit_id: nap.Ts(group["spike_time"].to_numpy(), time_units="s")
    for unit_id, group in df.groupby("unit_id")
}
tsgroup = nap.TsGroup(spikes)

387

We can also add metadata to represent the brain area. The series must be reindexed to match the unit ID indices of the TsGroup.

In [23]:
metadata = df[["unit_id", "brain_area"]].drop_duplicates().set_index("unit_id").reindex(spikes.keys())
metadata.head()

,brain_area
unit_id,
951015628,AM
951015643,AM
951015704,AM
951015717,AM
951015731,AM


In [25]:
tsgroup = nap.TsGroup(spikes, metadata=metadata)
tsgroup

Index      rate      brain_area
---------  --------  ------------
951015628  0.28331   AM
951015643  0.25295   AM
951015704  0.10455   AM
951015717  2.01689   AM
951015731  0.10118   AM
951015763  1.55482   AM
951015774  1.8921    AM
...        ...       ...
951039413  11.09963  RL
951039454  0.70153   RL
951039462  2.53966   RL
951039560  0.56662   RL
951039854  0.25633   RL
951039863  0.38449   RL
951039870  11.47737  RL